In [40]:
import pandas as pd
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import csv
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [41]:
all_files = glob.glob("*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {all_files}")

df.info()
df.head()

Total rows: 4111
Dari 1 file: ['linkedin_jobs_final_20260506_115218.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4111 entries, 0 to 4110
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4111 non-null   float64
 1   title              4111 non-null   object 
 2   standardized_role  4111 non-null   object 
 3   company            4111 non-null   object 
 4   location           4111 non-null   object 
 5   city               4111 non-null   object 
 6   work_type          4111 non-null   object 
 7   seniority          4111 non-null   object 
 8   jd_length          4111 non-null   int64  
 9   job_description    4111 non-null   object 
 10  extracted_skills   4111 non-null   object 
 11  skills_count       4111 non-null   int64  
 12  job_url            4111 non-null   object 
 13  search_role        4111 non-null   object 
 14  scraped_at         4111 non-null   object 
dty

,id,title,standardized_role,company,location,city,work_type,seniority,jd_length,job_description,extracted_skills,skills_count,job_url,search_role,scraped_at
0,4.354913e+09,Data Analyst,Data Analyst,ParagonCorp,"Jakarta, Indonesia (On-site)",Jakarta,On-site,Mid-level,1570,About the job\n\nJob Description\n\nDeliver in...,"['postgresql', 'power bi', 'python', 'sql', 't...",5,https://www.linkedin.com/jobs/view/4354913165/...,Data Scientist,2026-04-16T21:04:46.182549
1,4.395935e+09,Intern,Other,Schoters,Jakarta Metropolitan Area (Hybrid),Jakarta (Hybrid),Hybrid,Intern,999,About the job\n\nLooking for a chance to kicks...,[],0,https://www.linkedin.com/jobs/view/4395934526/...,Data Scientist,2026-04-16T21:04:50.857008
2,4.401866e+09,Data Scientist,Data Scientist,Artajasa Pembayaran Elektronis,"Kota Tangerang Selatan, Banten, Indonesia (On-...",Kota Tangerang Selatan,On-site,Mid-level,2138,About the job\n\nPT Artajasa Pembayaran Elektr...,"['numpy', 'pandas', 'python', 'r', 'scikit-lea...",6,https://www.linkedin.com/jobs/view/4401866027/...,Data Scientist,2026-04-16T21:04:55.554607
3,4.402503e+09,Data Scientist (Remote),Data Scientist,Jobs Ai,APAC (Remote),APAC (Remote),Remote,Mid-level,2305,About the job\n\nRole: Data Scientist (Remote)...,"['python', 'r', 'sql']",3,https://www.linkedin.com/jobs/view/4402503224/...,Data Scientist,2026-04-16T21:05:00.274453
4,4.367583e+09,Investment Banking Intern,Other,Swiss Finance Institute,APAC (Remote),APAC (Remote),Remote,Intern,6796,About the job\n\nInvestment Banking Internship...,[],0,https://www.linkedin.com/jobs/view/4367583109/...,Data Scientist,2026-04-16T21:05:09.748073


In [42]:
print(f"Jumlah unique role: {df['standardized_role'].nunique()}")
print()
print(df['standardized_role'].value_counts())

Jumlah unique role: 14

standardized_role
Other                  1712
Data Analyst            408
Software Engineer       369
Backend Developer       361
Data Engineer           318
Fullstack Developer     165
DevOps Engineer         161
ML Engineer             153
QA Engineer             136
Mobile Developer         74
Data Scientist           70
BI Developer             64
Frontend Developer       62
System Analyst           58
Name: count, dtype: int64


In [43]:
df = df.drop_duplicates()
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")
print(f"total null column : {df.isnull().sum()}")

total duplicated columns : 0
total null column : id                   0
title                0
standardized_role    0
company              0
location             0
city                 0
work_type            0
seniority            0
jd_length            0
job_description      0
extracted_skills     0
skills_count         0
job_url              0
search_role          0
scraped_at           0
dtype: int64


In [44]:
df.head()

,id,title,standardized_role,company,location,city,work_type,seniority,jd_length,job_description,extracted_skills,skills_count,job_url,search_role,scraped_at
0,4.354913e+09,Data Analyst,Data Analyst,ParagonCorp,"Jakarta, Indonesia (On-site)",Jakarta,On-site,Mid-level,1570,About the job\n\nJob Description\n\nDeliver in...,"['postgresql', 'power bi', 'python', 'sql', 't...",5,https://www.linkedin.com/jobs/view/4354913165/...,Data Scientist,2026-04-16T21:04:46.182549
1,4.395935e+09,Intern,Other,Schoters,Jakarta Metropolitan Area (Hybrid),Jakarta (Hybrid),Hybrid,Intern,999,About the job\n\nLooking for a chance to kicks...,[],0,https://www.linkedin.com/jobs/view/4395934526/...,Data Scientist,2026-04-16T21:04:50.857008
2,4.401866e+09,Data Scientist,Data Scientist,Artajasa Pembayaran Elektronis,"Kota Tangerang Selatan, Banten, Indonesia (On-...",Kota Tangerang Selatan,On-site,Mid-level,2138,About the job\n\nPT Artajasa Pembayaran Elektr...,"['numpy', 'pandas', 'python', 'r', 'scikit-lea...",6,https://www.linkedin.com/jobs/view/4401866027/...,Data Scientist,2026-04-16T21:04:55.554607
3,4.402503e+09,Data Scientist (Remote),Data Scientist,Jobs Ai,APAC (Remote),APAC (Remote),Remote,Mid-level,2305,About the job\n\nRole: Data Scientist (Remote)...,"['python', 'r', 'sql']",3,https://www.linkedin.com/jobs/view/4402503224/...,Data Scientist,2026-04-16T21:05:00.274453
4,4.367583e+09,Investment Banking Intern,Other,Swiss Finance Institute,APAC (Remote),APAC (Remote),Remote,Intern,6796,About the job\n\nInvestment Banking Internship...,[],0,https://www.linkedin.com/jobs/view/4367583109/...,Data Scientist,2026-04-16T21:05:09.748073


In [45]:
import ast

def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

print(f"Data setelah filter: {len(df)} rows")
print(f"Skills masih kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['standardized_role'].value_counts())

Data setelah filter: 4111 rows
Skills masih kosong: 1514 rows
standardized_role
Other                  1712
Data Analyst            408
Software Engineer       369
Backend Developer       361
Data Engineer           318
Fullstack Developer     165
DevOps Engineer         161
ML Engineer             153
QA Engineer             136
Mobile Developer         74
Data Scientist           70
BI Developer             64
Frontend Developer       62
System Analyst           58
Name: count, dtype: int64


In [46]:
min_samples = 200
role_counts = df['standardized_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['standardized_role'].isin(valid_roles)].reset_index(drop=True)
print(df['standardized_role'].value_counts())

standardized_role
Other                1712
Data Analyst          408
Software Engineer     369
Backend Developer     361
Data Engineer         318
Name: count, dtype: int64


In [47]:
from sklearn.utils import resample

max_samples = 800
dfs = []
for role, group in df.groupby('standardized_role'):
    if len(group) > max_samples:
        group = resample(group, n_samples=max_samples, random_state=42)
    dfs.append(group)

df = pd.concat(dfs).reset_index(drop=True)
print(df['standardized_role'].value_counts())

standardized_role
Other                800
Data Analyst         408
Software Engineer    369
Backend Developer    361
Data Engineer        318
Name: count, dtype: int64


In [48]:
df['text_input'] = df['job_description'] + ' ' + df['extracted_skills_clean']

role_mapping = {
    'software':           'Software Engineer',
    'Software':           'Software Engineer',
    'Software Developer': 'Software Engineer',
    'Developer':          'Software Engineer',
    'Progammer':          'Software Engineer',
    'Full stack Developer': 'Fullstack Developer',
    'Web Developer':      'Fullstack Developer',
    'IT':                 'Software Engineer',
}


df['standardized_role'] = df['standardized_role'].replace(role_mapping)
print(df['standardized_role'].value_counts())

df = df[df['standardized_role'] != 'Software Engineer']
df = df[df['standardized_role'] != 'Other']

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['standardized_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

df['text_input'] = df['text_input'].fillna('').astype(str)
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

max_vocab_length = 20000
max_sequence_length = 256

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

standardized_role
Other                800
Data Analyst         408
Software Engineer    369
Backend Developer    361
Data Engineer        318
Name: count, dtype: int64
Total role unik: 3
Mapping kelas:
{'Backend Developer': 0, 'Data Analyst': 1, 'Data Engineer': 2}

Shape X_train: (869,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[  22    4   19  195 1916  709  149 2122  322 1814  392  759  287 5498
   299  178  367 1470  987   39   95   44  157    2 1227 1916  709    2
  2108  328   69 2122   63 1062   46    2 2478   40   27   93 1420    2
  1225  542   82    8  462 4303    2 4016    3  300   68   34   74  380
     2  785   85  163  103   48   64  282  804 2672    2  484  313  266
     5    4  535   29  605   72 2915   29 2416    2  699  441  376   37
   227  593   91  208    2  845    8   26  412  691  931   20 2122 2703
     2 1014  104  114   34   25  183   12    8 2122   10 1916  709   11
  2627  103   29  100    5   51   11  334   10 2478  625 1225    2 1401
  1090 

In [49]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.core.series.Series'> object
179    About the job\n\nPosition: 3D Game Developer (...
98     About the job\n\nDEVELOPER\n\nJUNIOR\n\nINTERN...
141    About the job\n\nWe Are Hiring\n\n\nNET Develo...
Name: text_input, dtype: object
object


In [50]:
import numpy as np

class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3):
        super(CustomEarlyStopping, self).__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Overfitting terdeteksi. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

In [51]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.4)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier_LSTM")

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMulai proses training...")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[CustomEarlyStopping(patience=7)]
)



Mulai proses training...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.3959 - loss: 1.0883 - val_accuracy: 0.4450 - val_loss: 1.0764
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5570 - loss: 1.0587 - val_accuracy: 0.5734 - val_loss: 1.0487
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6260 - loss: 1.0232 - val_accuracy: 0.6835 - val_loss: 1.0043
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7054 - loss: 0.9532 - val_accuracy: 0.7202 - val_loss: 0.9392
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7998 - loss: 0.8604 - val_accuracy: 0.7569 - val_loss: 0.8437
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8354 - loss: 0.7363 - val_accuracy: 0.7844 - val_loss: 0.7329
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8516 - loss: 0.6051 - val_accuracy: 0.7844 - val_loss: 0.6354
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8964 - loss: 0.4773 

In [56]:
model_path = 'job_role_model.keras'
model.save(model_path)
print(f"[INFO] Model berhasil diekspor ke: {model_path}")

loaded_model = tf.keras.models.load_model(model_path)

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)

    pred_probs = loaded_model.predict(input_tensor, verbose=0)

    pred_class_idx = np.argmax(pred_probs)

    predicted_role = encoder.inverse_transform([pred_class_idx])[0]

    return predicted_role

# 3. Test fungsi inference
skill_input = "i have experience for applying AI in machine learning, deep learning, Tensorflow, Python, NLP. Understanding of functional design principles, object-oriented programming principles, basic algorithms. ⁠Expertise in REST API development, NoSQL design, RDBMS design"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil diekspor ke: job_role_model.keras

--- Hasil Inference ---
Skill User : i have experience for applying AI in machine learning, deep learning, Tensorflow, Python, NLP. Understanding of functional design principles, object-oriented programming principles, basic algorithms. ⁠Expertise in REST API development, NoSQL design, RDBMS design
Cocok untuk: Backend Developer
